In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

In [8]:
def backtest(data, stopLoss, takeProfit):
    total_value = 10000
    returns = []
    holdingDays = []

    for row in data.itertuples():
        acquisition_price = getattr(row, 'p2', None)
        if acquisition_price is None or pd.isna(acquisition_price):
            continue

        profit_loss = None
        daysHeld = 19

        for i in range(2,21):
            price = getattr(row, f"p{i}", None)
            if price is None or pd.isna(price):
                continue
            if price <= acquisition_price*(1-stopLoss):
                profit_loss = -stopLoss
                daysHeld = i-1
                break
            elif price >= acquisition_price*(1+takeProfit):
                profit_loss = takeProfit
                daysHeld = i-1
                break
        if profit_loss is None:
            price_Day20 = getattr(row, "p20", None)
            if price_Day20 is not None and not pd.isna(price_Day20):
                profit_loss = (price_Day20 - acquisition_price) / acquisition_price
            else:
                profit_loss = 0 
             
        returns.append(profit_loss)
        holdingDays.append(daysHeld)
        total_value =   total_value + 10000 * profit_loss

    if (len(returns)==0):
        return {
        "Total Return": 0,
        "Sharpe Ratio": -np.inf,
        "Win Rate": 0,
        "Average Holding Days": 0,
        "Profit Factor": 0,
        "Total Trades": 0,
        }

    returns = np.array(returns)
    holdingDays = np.array(holdingDays)

    avg_returns = np.mean(returns)
    std_returns = np.std(returns)
    sharpe = (avg_returns/std_returns) if std_returns > 0 else -np.inf
    # totalPercReturn = (total_value - 10000)/ 10000
    totalPercReturn = np.sum(returns)

    wins = returns > 0
    losses = returns < 0
    winRate = np.mean(wins)
    avgHoldingDays = np.mean(holdingDays)

    totalGains = np.sum(returns[wins])
    totalLosses = - np.sum(returns[losses])
    profitFactor = (totalGains / totalLosses) if totalLosses > 0 else np.inf


    results = {
        "Total Return": totalPercReturn,
        "Sharpe Ratio": sharpe,
        "Win Rate": winRate,
        "Average Holding Days": avgHoldingDays,
        "Profit Factor": profitFactor,
        "Total Trades": len(returns),
    }

    return results

In [9]:
def optimal_triggers(data, n_splits = 4):
    data = data[data['NCAR20_quintiles'].astype(float) == 1]
    print(data.shape)
    stopLoss_values = np.round(np.arange(0.05, 0.35, 0.01),2)
    takeProfit_values = np.round(np.arange(0.05, 0.60, 0.01),2)

    k_fold = KFold(n_splits, shuffle = False)

    results = []

    for sL in stopLoss_values:
        for tP in takeProfit_values:
            sharpes_values = []
            returns_values =[]
            winRate_values = []
            holdingDays_values = []
            profitFactor_values = []
            total_trades = 0

            for tr_indx, val_indx in k_fold.split(data):
                val_data = data.iloc[val_indx]
                r = backtest(val_data, sL, tP)
                sharpes_values.append(r['Sharpe Ratio'])
                returns_values.append(r['Total Return'])
                winRate_values.append(r['Win Rate'])
                holdingDays_values.append(r['Average Holding Days'])
                profitFactor_values.append(r['Profit Factor'])
                sharpes_values.append(r['Sharpe Ratio'])
                total_trades += r['Total Trades']

            results.append({'StopLoss': sL, 
                            'TakeProfit': tP, 
                            'TotalReturn': np.mean(returns_values) if returns_values else 0, 
                            'CV_SharpeRatio': np.mean(sharpes_values) if sharpes_values else -np.inf, 
                            'WinRate': np.mean(winRate_values) if winRate_values else 0, 
                            'AvgDays': np.mean(holdingDays_values) if holdingDays_values else 0,
                            'ProfitFactor': np.mean(profitFactor_values) if profitFactor_values else 0,
                            'TradeCount': total_trades
                            })
            
    results_df = pd.DataFrame(results)
    optimal_idx = results_df['CV_SharpeRatio'].idxmax()
    optimal_row = results_df.loc[optimal_idx]

    optimal_results = {
        'OptimalProfit': optimal_row['TotalReturn'], 
        'Optimal StopLoss': optimal_row['StopLoss'], 
        'Optimal TakeProfit': optimal_row['TakeProfit'], 
        'Optimal Sharpe': optimal_row['CV_SharpeRatio'], 
        'Optimal WinRate': optimal_row['WinRate'], 
        'Optimal AvgDays': optimal_row['AvgDays']
    }

    return optimal_results, results_df

In [10]:
final_df = pd.read_csv("datasets/final_ds.csv")
final_df = final_df[(final_df['date'] >= '2025-01-01') & (final_df['date'] <= '2025-11-30')]

prms, grid = optimal_triggers(final_df)
optimal_SL = prms['Optimal StopLoss']
optimal_TP = prms['Optimal TakeProfit']
optimal_Sharpe = prms['Optimal Sharpe']
print(optimal_SL, optimal_TP, optimal_Sharpe)

(463, 63)
0.28 0.4 0.680390957601113


In [15]:
final_df = pd.read_csv("datasets/final_ds.csv")
final_df_test = final_df[(final_df['date'] >= '2026-01-01') & (final_df['date'] <= '2026-6-30')]
final_df_test_filtered = final_df_test[final_df_test["NCAR20_quintiles"].astype(float) ==1]
backtest(final_df_test_filtered, optimal_SL, optimal_TP)

{'Total Return': np.float64(39.03521844921839),
 'Sharpe Ratio': np.float64(0.6413354593459668),
 'Win Rate': np.float64(0.7423728813559322),
 'Average Holding Days': np.float64(16.71864406779661),
 'Profit Factor': np.float64(5.0803262410634344),
 'Total Trades': 295}

In [16]:
final_df = pd.read_csv("datasets/test_data_with_preds.csv")
final_df_test = final_df[(final_df['date'] >= '2026-01-01') & (final_df['date'] <= '2026-6-30')]
final_df_test_filtered = final_df_test[final_df_test["NCAR20_preds_quintiles"].astype(float) ==1]
backtest(final_df_test_filtered, optimal_SL, optimal_TP)


{'Total Return': np.float64(-0.02799764407307337),
 'Sharpe Ratio': np.float64(-0.0004860827416533816),
 'Win Rate': np.float64(0.46621621621621623),
 'Average Holding Days': np.float64(17.6722972972973),
 'Profit Factor': np.float64(0.9987645602907244),
 'Total Trades': 296}